<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session03-discussion-1.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 3: in-class discussion problem (1 of 3)

This is the block 1 question of the lecture quiz. First guess the answer in the quiz, on your own.
Then run the code below and look at what it does. There is one mistake in it, but finding that is
only the way in: the point is to work out why the result comes out as it does. Discuss that with
your neighbour, then answer the quiz again.

## Same accession, new version: what changed?

A paper from 2015 used the TP53 mRNA record `NM_000546.5`. Today RefSeq has `NM_000546.6`. What can
you say about the two records?

- Is it the same gene, and the same record?
- Is it the same sequence?
- Does the protein change?

**Fix and run.** The first cell installs Biopython. The second loads both versions, as downloaded
from NCBI on 14 October 2026, and compares them, but the line marked `FIX` compares the wrong
thing. Run it once as it is: what does it claim? Then fix the line so that it compares the two
sequences, and run it again.

In [1]:
# Install Biopython if it is missing (Colab does not come with it)
import importlib.util, subprocess, sys
if importlib.util.find_spec("Bio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "biopython"], check=True)

In [2]:
import io, os, requests
from Bio import SeqIO

URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/session03-tp53-versions.fasta"
LOCAL = "data/session03-tp53-versions.fasta"
text = open(LOCAL).read() if os.path.exists(LOCAL) else requests.get(URL, timeout=30).text
seqs = {r.id: r.seq for r in SeqIO.parse(io.StringIO(text), "fasta")}

old, new = "NM_000546.5", "NM_000546.6"
for acc in (old, new):
    print(f"{acc}: {len(seqs[acc])} bases")

same = old.split(".")[0] == new.split(".")[0]      # FIX: this compares the accessions, not the sequences
print("same sequence?", same)

NM_000546.5: 2591 bases
NM_000546.6: 2512 bases
same sequence? True


**If you finish early.**

1. Is the new sequence part of the old one? Try `str(seqs[new]) in str(seqs[old])`, and find where
   it starts with `.find()`.
2. Translate both from the start codon: `s[s.find("ATGGAGGAGCCGCAG"):].translate(to_stop=True)`. Is
   the protein the same?

<details>
<summary><b>Discussion point</b> &mdash; open this once you have run the code and talked it through</summary>

As it was, the cell compared the accessions without their versions,
`NM_000546` with `NM_000546`, and claimed the sequences were the same. Comparing the sequences
themselves,

```python
same = seqs[old] == seqs[new]
```

gives `False`: version 5 has 2591 bases, version 6 has 2512. It is still the same gene and the same
record (the accession before the dot), but **the number after the dot is the sequence version**: it
goes up only when the sequence itself changes. Here version 6 is version 5 trimmed by 60 bases at the
5′ end and 19 at the 3′ end, in the untranslated regions. The coding sequence is untouched, so both
translate to the same 393-residue p53 protein (`NP_000537.3`).

The habit: always record the full accession with its version. Positions in the 2015 paper ("base
200 of NM_000546") refer to version 5; in version 6 the same base is number 140.

</details>